# Vision-Language Models (VLMs) from Scratch

**Lecture 2, Part 5 — Vizuara Modern Robot Learning Bootcamp**

We build a **Vision-Language Model** from scratch, bridging from ViT (Part 4) to pi0 (next lecture):

| Component | What it does | Key idea |
|-----------|-------------|----------|
| SigLIP | Align image and text embeddings | Contrastive learning with sigmoid loss |
| Projection Bridge | Map ViT dim → LLM dim | Single linear layer translates coordinate systems |
| VLM Assembly | Visual tokens + text tokens → one sequence | Cross-modal attention in a shared Transformer |
| Cross-Modal Attention | Text words attend to image patches | "red" → red pixels, "cup" → cup patches |
| PaliGemma | The real VLM behind pi0 | SigLIP-So400M + Gemma 2B = 2.4B params |

By the end, you'll build a working mini-VLM and understand PaliGemma — the foundation behind pi0.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from sklearn.decomposition import PCA
import math

# Vizuara color palette
ACCENT = '#D97757'
BLUE   = '#5B8CB8'
TEAL   = '#7DA488'
PURPLE = '#9B7EC8'
WARM   = '#C4956A'
BG     = '#FAF8F5'

plt.rcParams['figure.facecolor'] = BG
plt.rcParams['axes.facecolor'] = BG
plt.rcParams['font.size'] = 12

torch.manual_seed(42)
np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

---
## Step 1: The Problem — Vision and Language Live in Different Worlds

In Part 4 we built a ViT that turns images into patch tokens. In Part 3 we built a Transformer that turns text into word tokens. But these two systems know **nothing about each other**.

A ViT trained on ImageNet encodes "redness" as some direction in 768-d space. A language model encodes the word "red" as some completely unrelated direction in 2048-d space. There's no reason these should be aligned.

This is the fundamental problem VLMs solve: **bridge the gap between vision and language.**

In [ ]:
def create_robot_scene(size=224):
    """Create a synthetic robot workspace image with colored objects."""
    img = np.ones((size, size, 3), dtype=np.float32) * 0.85  # light gray table

    # Brown table edge (bottom third)
    img[size*2//3:, :] = [0.55, 0.35, 0.2]

    # Red cup (circle)
    cy, cx, r = size//3, size//4, size//10
    yy, xx = np.ogrid[:size, :size]
    mask = (xx - cx)**2 + (yy - cy)**2 < r**2
    img[mask] = [0.85, 0.2, 0.15]

    # Blue plate (ellipse)
    cy2, cx2 = size//3, size*3//4
    mask2 = ((xx - cx2)/1.5)**2 + ((yy - cy2)/1.0)**2 < (r*0.8)**2
    img[mask2] = [0.2, 0.4, 0.75]

    # Green cube
    s = size // 12
    y0, x0 = size//2 - s//2, size//2 - s//2
    img[y0:y0+s, x0:x0+s] = [0.2, 0.65, 0.35]

    # Robot gripper (gray arm from top)
    gw = size // 20
    gx = size * 3 // 5
    img[:size//4, gx-gw:gx+gw] = [0.4, 0.4, 0.45]  # arm
    # Gripper tips
    img[size//4-5:size//4+5, gx-gw*2:gx-gw] = [0.35, 0.35, 0.4]
    img[size//4-5:size//4+5, gx+gw:gx+gw*2] = [0.35, 0.35, 0.4]

    return np.clip(img, 0, 1)

scene = create_robot_scene(224)

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(scene)
ax.set_title('Robot Workspace (224x224)', fontsize=14, color=ACCENT, fontweight='bold')
ax.axis('off')

# Label objects
ax.annotate('Red Cup', (56, 50), fontsize=11, color='white', fontweight='bold',
            bbox=dict(boxstyle='round', fc=ACCENT, alpha=0.8))
ax.annotate('Blue Plate', (148, 50), fontsize=11, color='white', fontweight='bold',
            bbox=dict(boxstyle='round', fc=BLUE, alpha=0.8))
ax.annotate('Green Cube', (95, 125), fontsize=11, color='white', fontweight='bold',
            bbox=dict(boxstyle='round', fc=TEAL, alpha=0.8))
ax.annotate('Gripper', (140, 30), fontsize=11, color='white', fontweight='bold',
            bbox=dict(boxstyle='round', fc='gray', alpha=0.8))

plt.tight_layout()
plt.show()

print(f"Scene shape: {scene.shape}")
print(f"This image will be our running example throughout the notebook.")

In [ ]:
# Demonstrate the gap: image and text embeddings live in separate worlds
D_EMBED = 64
N_SAMPLES = 5

# Toy image embeddings (what a ViT might produce — random, unaligned)
image_labels = ['red cup', 'blue plate', 'green cube', 'gripper', 'table']
text_labels  = ['red cup', 'blue plate', 'green cube', 'gripper', 'table']

torch.manual_seed(42)
image_embeds = torch.randn(N_SAMPLES, D_EMBED)  # ViT space
text_embeds  = torch.randn(N_SAMPLES, D_EMBED)  # LLM space

# PCA to 2D for visualization
all_embeds = torch.cat([image_embeds, text_embeds]).numpy()
pca = PCA(n_components=2)
all_2d = pca.fit_transform(all_embeds)
img_2d = all_2d[:N_SAMPLES]
txt_2d = all_2d[N_SAMPLES:]

# Cosine similarity matrix
img_norm = F.normalize(image_embeds, dim=-1)
txt_norm = F.normalize(text_embeds, dim=-1)
sim_matrix = (img_norm @ txt_norm.T).numpy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: PCA scatter
ax = axes[0]
for i in range(N_SAMPLES):
    ax.scatter(*img_2d[i], color=ACCENT, s=120, zorder=5, marker='s')
    ax.annotate(f'IMG: {image_labels[i]}', img_2d[i], fontsize=9,
                xytext=(8, 5), textcoords='offset points', color=ACCENT)
    ax.scatter(*txt_2d[i], color=BLUE, s=120, zorder=5, marker='^')
    ax.annotate(f'TXT: {text_labels[i]}', txt_2d[i], fontsize=9,
                xytext=(8, 5), textcoords='offset points', color=BLUE)
ax.set_title('Embedding Space (PCA)', fontsize=13, color=ACCENT, fontweight='bold')
ax.legend(['Image embeds', 'Text embeds'], fontsize=10)
ax.grid(True, alpha=0.3)

# Right: Cosine similarity heatmap
ax = axes[1]
im = ax.imshow(sim_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(N_SAMPLES))
ax.set_yticks(range(N_SAMPLES))
ax.set_xticklabels(text_labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(image_labels, fontsize=9)
ax.set_xlabel('Text embeddings', fontsize=11)
ax.set_ylabel('Image embeddings', fontsize=11)
for i in range(N_SAMPLES):
    for j in range(N_SAMPLES):
        ax.text(j, i, f'{sim_matrix[i, j]:.2f}', ha='center', va='center', fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('Image-Text Cosine Similarity', fontsize=13, color=BLUE, fontweight='bold')

plt.suptitle('The Gap: Vision and Language Are Unaligned',
             fontsize=15, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

print("The ViT sees shapes. The LLM reads words. Neither understands the other.")
print(f"\nCosine similarity between matching pairs:")
for i in range(N_SAMPLES):
    print(f"  IMG '{image_labels[i]}' <-> TXT '{text_labels[i]}': {sim_matrix[i, i]:.3f}")
print(f"\nDiagonal (matching pairs) should be ~1.0, but it's random noise.")
print(f"We need a way to ALIGN these two spaces. That's what SigLIP does.")

---
## Step 2: SigLIP — Aligning Vision and Language with Contrastive Learning

**SigLIP** (Sigmoid Loss for Language-Image Pre-training) learns to pull matching image-text pairs together and push non-matching pairs apart.

The core idea:
- Given a batch of N image-text pairs, compute an N×N similarity matrix
- **Diagonal entries** (matching pairs) should be high → push toward +1
- **Off-diagonal entries** (non-matching pairs) should be low → push toward -1
- SigLIP uses **sigmoid per-pair** instead of CLIP's softmax per-row, which works better with smaller batch sizes

This is how models like SigLIP, CLIP, and OpenCLIP learn to connect vision and language.

In [ ]:
# Toy contrastive dataset: 8 image-caption pairs
N_PAIRS = 8
D_RAW = 64  # raw embedding dimension (before projection)

captions = [
    'red cup on table',
    'blue plate near edge',
    'green cube center',
    'robot gripper open',
    'yellow ball rolling',
    'white bowl upside down',
    'orange cone standing',
    'purple block stacked',
]

torch.manual_seed(42)
# Random "raw" embeddings — as if from a frozen ViT and frozen text encoder
raw_image_embeds = torch.randn(N_PAIRS, D_RAW)
raw_text_embeds  = torch.randn(N_PAIRS, D_RAW)

# Show similarity matrix BEFORE training (chaotic)
img_n = F.normalize(raw_image_embeds, dim=-1)
txt_n = F.normalize(raw_text_embeds, dim=-1)
sim_before = (img_n @ txt_n.T).detach().numpy()

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(sim_before, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(N_PAIRS))
ax.set_yticks(range(N_PAIRS))
short_labels = [c.split()[0] + ' ' + c.split()[1] for c in captions]
ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(short_labels, fontsize=9)
ax.set_xlabel('Text embeddings', fontsize=11)
ax.set_ylabel('Image embeddings', fontsize=11)
for i in range(N_PAIRS):
    for j in range(N_PAIRS):
        ax.text(j, i, f'{sim_before[i, j]:.2f}', ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('Image-Text Similarity BEFORE Training (chaotic)',
             fontsize=13, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"8 image-text pairs, each {D_RAW}-dimensional.")
print(f"The diagonal should be bright (matching pairs), but it's random noise.")
print(f"SigLIP will fix this.")

In [ ]:
def siglip_loss(image_embeds, text_embeds, temperature=1.0):
    """SigLIP contrastive loss: sigmoid per-pair instead of softmax per-row.

    For each (i, j) pair:
      - If i == j (matching):   target = +1, push similarity UP
      - If i != j (non-match):  target = -1, push similarity DOWN
      - Loss = -log(sigmoid(target * similarity / temperature))

    We balance positive and negative contributions so that the 8 matching
    pairs count equally to the 56 non-matching pairs (otherwise the model
    collapses all similarities to -1).

    Why sigmoid > softmax (CLIP)?
      - Softmax couples all pairs in a row → needs huge batch sizes (32K+)
      - Sigmoid treats each pair independently → works with smaller batches
    """
    # Normalize embeddings
    img_n = F.normalize(image_embeds, dim=-1)
    txt_n = F.normalize(text_embeds, dim=-1)

    # Similarity matrix: [N, N]
    logits = (img_n @ txt_n.T) / temperature

    # Target matrix: +1 on diagonal, -1 off-diagonal
    N = image_embeds.shape[0]
    targets = 2 * torch.eye(N) - 1  # [[+1, -1, -1, ...], [-1, +1, -1, ...], ...]

    # Sigmoid loss per pair
    per_pair = -F.logsigmoid(targets * logits)

    # Balance positive and negative contributions
    pos_mask = torch.eye(N).bool()
    loss_pos = per_pair[pos_mask].mean()    # N matching pairs
    loss_neg = per_pair[~pos_mask].mean()   # N*(N-1) non-matching pairs
    return (loss_pos + loss_neg) / 2

# Test the loss
loss_val = siglip_loss(raw_image_embeds, raw_text_embeds, temperature=1.0)
print(f"SigLIP loss (before training): {loss_val.item():.4f}")
print(f"\nThe loss function:")
print(f"  For matching pair (i, i):   -log(sigmoid(+sim / T))  → wants sim HIGH")
print(f"  For non-match (i, j!=i):    -log(sigmoid(-sim / T))  → wants sim LOW")
print(f"  Balanced: positive and negative contributions weighted equally")
print(f"\nWhy SigLIP over CLIP?")
print(f"  CLIP uses softmax per-row → all N pairs compete → needs N=32,768")
print(f"  SigLIP uses sigmoid per-pair → each pair is independent → works with N=8!")

In [ ]:
# Train SigLIP: learn projection heads that align image and text spaces
torch.manual_seed(42)

D_PROJ = 64  # projection output dimension

# Two learnable projection heads (lightweight MLPs)
image_proj = nn.Sequential(
    nn.Linear(D_RAW, 128), nn.ReLU(), nn.Linear(128, D_PROJ)
)
text_proj = nn.Sequential(
    nn.Linear(D_RAW, 128), nn.ReLU(), nn.Linear(128, D_PROJ)
)

optimizer = torch.optim.Adam(
    list(image_proj.parameters()) + list(text_proj.parameters()), lr=1e-2
)

# Training loop
N_STEPS = 500
losses = []
sim_snapshots = {}  # save similarity matrices at key steps
snapshot_steps = [0, 150, 300, 500]

for step in range(N_STEPS + 1):
    img_projected = image_proj(raw_image_embeds)
    txt_projected = text_proj(raw_text_embeds)

    loss = siglip_loss(img_projected, txt_projected, temperature=1.0)
    losses.append(loss.item())

    # Save snapshots
    if step in snapshot_steps:
        with torch.no_grad():
            i_n = F.normalize(img_projected, dim=-1)
            t_n = F.normalize(txt_projected, dim=-1)
            sim_snapshots[step] = (i_n @ t_n.T).numpy()

    if step < N_STEPS:
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

# Plot: loss curve + similarity matrices at 4 checkpoints
fig, axes = plt.subplots(1, 5, figsize=(26, 5))

# Loss curve
ax = axes[0]
ax.plot(losses, color=ACCENT, linewidth=2)
ax.set_xlabel('Step', fontsize=11)
ax.set_ylabel('SigLIP Loss', fontsize=11)
ax.set_title('Training Loss', fontsize=13, color=ACCENT, fontweight='bold')
ax.grid(True, alpha=0.3)
for s in snapshot_steps:
    ax.axvline(s, color='gray', linestyle='--', alpha=0.3)
    ax.text(s, max(losses)*0.9, f'{s}', fontsize=8, ha='center')

# Similarity matrices at 4 checkpoints
step_colors = [ACCENT, BLUE, TEAL, PURPLE]
for idx, (step, c) in enumerate(zip(snapshot_steps, step_colors)):
    ax = axes[idx + 1]
    sim = sim_snapshots[step]
    im = ax.imshow(sim, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_title(f'Step {step}', fontsize=12, color=c, fontweight='bold')
    ax.set_xticks(range(N_PAIRS))
    ax.set_yticks(range(N_PAIRS))
    ax.set_xticklabels(range(N_PAIRS), fontsize=8)
    ax.set_yticklabels(range(N_PAIRS), fontsize=8)
    for i in range(N_PAIRS):
        for j in range(N_PAIRS):
            val = sim[i, j]
            color = 'white' if abs(val) > 0.5 else 'black'
            ax.text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=7, color=color)

plt.suptitle('SigLIP Training: From Chaos to Clean Diagonal',
             fontsize=15, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Loss: {losses[0]:.3f} (step 0) → {losses[-1]:.3f} (step {N_STEPS})")
print(f"\nStep 0:   Random noise — no alignment")
print(f"Step 150: Diagonal emerging — matching pairs getting brighter")
print(f"Step 300: Clear diagonal — off-diagonal pushed negative")
print(f"Step 500: Clean identity matrix — perfect alignment!")

In [ ]:
# Embedding space visualization: BEFORE vs AFTER training
with torch.no_grad():
    img_after = image_proj(raw_image_embeds)
    txt_after = text_proj(raw_text_embeds)

# PCA: before training (raw embeddings)
all_before = torch.cat([raw_image_embeds, raw_text_embeds]).numpy()
pca_before = PCA(n_components=2)
coords_before = pca_before.fit_transform(all_before)

# PCA: after training (projected embeddings)
all_after = torch.cat([img_after, txt_after]).numpy()
pca_after = PCA(n_components=2)
coords_after = pca_after.fit_transform(all_after)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

pair_colors = [ACCENT, BLUE, TEAL, PURPLE, WARM, '#D4543A', '#666666', '#4A90A4']

for ax_idx, (ax, coords, title) in enumerate([
    (axes[0], coords_before, 'BEFORE Training (random)'),
    (axes[1], coords_after, 'AFTER SigLIP Training'),
]):
    for i in range(N_PAIRS):
        c = pair_colors[i]
        # Image point (square)
        ax.scatter(coords[i, 0], coords[i, 1], color=c, s=150, marker='s',
                   edgecolors='black', linewidths=0.5, zorder=5)
        # Text point (triangle)
        ax.scatter(coords[N_PAIRS + i, 0], coords[N_PAIRS + i, 1], color=c, s=150,
                   marker='^', edgecolors='black', linewidths=0.5, zorder=5)
        # Line connecting matching pair
        ax.plot([coords[i, 0], coords[N_PAIRS + i, 0]],
                [coords[i, 1], coords[N_PAIRS + i, 1]],
                color=c, alpha=0.4, linewidth=1, linestyle='--')
        if ax_idx == 1:  # only label in the "after" plot to reduce clutter
            mid_x = (coords[i, 0] + coords[N_PAIRS + i, 0]) / 2
            mid_y = (coords[i, 1] + coords[N_PAIRS + i, 1]) / 2
            ax.annotate(short_labels[i], (mid_x, mid_y), fontsize=7,
                        ha='center', color=c, fontweight='bold')

    ax.set_title(title, fontsize=13, color=ACCENT if ax_idx == 0 else TEAL, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(['Square = Image', 'Triangle = Text'], fontsize=9, loc='upper right')

plt.suptitle('SigLIP Aligns Matching Image-Text Pairs in Embedding Space',
             fontsize=15, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

# Compare diagonal similarity before vs after (from the snapshot matrices)
first_step = snapshot_steps[0]
last_step = snapshot_steps[-1]
print("Cosine similarity for matching pairs (diagonal of similarity matrix):")
print(f"  {'Pair':>16s}   {'Before':>8s}   {'After':>8s}")
print(f"  {'-'*40}")
for i in range(N_PAIRS):
    sim_b = sim_snapshots[first_step][i, i]
    sim_a = sim_snapshots[last_step][i, i]
    print(f"  {short_labels[i]:>16s}   {sim_b:+.3f}    {sim_a:+.3f}  {'↑' if sim_a > sim_b else '↓'}")

diag_before = np.diag(sim_snapshots[first_step]).mean()
diag_after = np.diag(sim_snapshots[last_step]).mean()
print(f"\n  Mean diagonal: {diag_before:+.3f} → {diag_after:+.3f}")
print(f"\nSigLIP has aligned vision and language.")
print(f"'Red cup' image <-> 'red cup' text now have high cosine similarity.")

---
## Step 3: The Projection Bridge

SigLIP aligns vision and language in a **shared** embedding space. But in a full VLM, the ViT and LLM have different hidden dimensions:

- **SigLIP-So400M** (ViT): 1152-d per patch token
- **Gemma 2B** (LLM): 2048-d per text token

A single **linear projection** bridges the gap. That's it — just a matrix multiply. The ViT's features are already rich enough that a simple linear map can translate them into the LLM's coordinate system.

In [ ]:
# The projection bridge: ViT dim → LLM dim
D_VIT = 1152       # SigLIP-So400M output dimension
D_LLM = 2048       # Gemma 2B embedding dimension
N_VISUAL_TOKENS = 196  # 14x14 patches from 224x224 image

projection = nn.Linear(D_VIT, D_LLM)

# Simulate ViT output
vit_tokens = torch.randn(1, N_VISUAL_TOKENS, D_VIT)

with torch.no_grad():
    llm_tokens = projection(vit_tokens)  # [1, 196, 2048]

proj_params = sum(p.numel() for p in projection.parameters())

print(f"Projection Bridge:")
print(f"  ViT output:       {list(vit_tokens.shape)} ({D_VIT}-d per token)")
print(f"  After projection: {list(llm_tokens.shape)} ({D_LLM}-d per token)")
print(f"  Parameters: {proj_params:,} ({D_VIT} x {D_LLM} + {D_LLM} bias)")
print(f"\nThat's {proj_params/1e6:.1f}M params — just 0.1% of PaliGemma's 2.4B total.")

# 3-panel visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# ViT space
ax = axes[0]
ax.imshow(vit_tokens[0, :16, :32].numpy(), aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
ax.set_title(f'ViT Space ({D_VIT}-d)', fontsize=13, color=ACCENT, fontweight='bold')
ax.set_xlabel('Dims (first 32)', fontsize=10)
ax.set_ylabel('Patches (first 16)', fontsize=10)

# Projection (weight matrix)
ax = axes[1]
W = projection.weight.detach()[:32, :32].numpy()  # show 32x32 corner
ax.imshow(W, aspect='auto', cmap='RdBu_r', vmin=-0.05, vmax=0.05)
ax.set_title(f'Projection W ({D_VIT}x{D_LLM})', fontsize=13, color=TEAL, fontweight='bold')
ax.set_xlabel('ViT dims', fontsize=10)
ax.set_ylabel('LLM dims', fontsize=10)

# LLM space
ax = axes[2]
ax.imshow(llm_tokens[0, :16, :32].numpy(), aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
ax.set_title(f'LLM Space ({D_LLM}-d)', fontsize=13, color=BLUE, fontweight='bold')
ax.set_xlabel('Dims (first 32)', fontsize=10)
ax.set_ylabel('Patches (first 16)', fontsize=10)

plt.suptitle('Projection Bridge: ViT Space → LLM Space',
             fontsize=15, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nWhy not a deep MLP? The ViT's features are already rich.")
print(f"A linear map just 'translates' them into the LLM's coordinate system.")

---
## Step 4: VLM Assembly — One Sequence, Two Modalities

This is the core VLM idea: **concatenate** visual tokens and text tokens into one sequence, then let the Transformer do cross-modal reasoning via self-attention.

```
[img_patch_0, img_patch_1, ..., img_patch_195, "pick", "up", "the", "red", "cup", "from", "table"]
```

That's 196 + 7 = **203 tokens** in one unified sequence.

In [ ]:
# VLM assembly: concatenate visual + text tokens
N_TEXT = 7
text_words = ["pick", "up", "the", "red", "cup", "from", "table"]

# Simulated tokens (already in LLM space)
visual_tokens = torch.randn(1, N_VISUAL_TOKENS, D_LLM)  # [1, 196, 2048]
text_tokens   = torch.randn(1, N_TEXT, D_LLM)            # [1, 7, 2048]

# Concatenate
vlm_input = torch.cat([visual_tokens, text_tokens], dim=1)  # [1, 203, 2048]
N_TOTAL = N_VISUAL_TOKENS + N_TEXT

print(f"VLM Input Assembly:")
print(f"  Visual tokens: {list(visual_tokens.shape)} (196 image patches from 14x14 grid)")
print(f"  Text tokens:   {list(text_tokens.shape)} ({N_TEXT} words)")
print(f"  Combined:      {list(vlm_input.shape)} (one unified sequence)")
print(f"")
print(f"Sequence layout:")
print(f"  Positions 0-195:   [img_0] [img_1] ... [img_195]  (14x14 patch grid)")
print(f"  Positions 196-202: [{'] ['.join(text_words)}]")
print(f"")
print(f"Now self-attention connects EVERYTHING:")
print(f"  'red' (pos 199) <-> red-colored patches (image)")
print(f"  'cup' (pos 200) <-> cup-shaped patches (image)")
print(f"  'pick' (pos 196) <-> gripper patches (image)")

In [ ]:
# VLM attention mask: visual = bidirectional, text = causal, text->visual = allowed
mask = torch.zeros(N_TOTAL, N_TOTAL)

# Visual tokens: attend to all visual tokens (bidirectional)
mask[:N_VISUAL_TOKENS, :N_VISUAL_TOKENS] = 1.0

# Text tokens: causal attention among themselves
for i in range(N_TEXT):
    for j in range(i + 1):
        mask[N_VISUAL_TOKENS + i, N_VISUAL_TOKENS + j] = 1.0

# Text tokens: can attend to ALL visual tokens
mask[N_VISUAL_TOKENS:, :N_VISUAL_TOKENS] = 1.0

# Visual tokens: cannot attend to text tokens (prefix-style)
# (already 0 by default)

# Visualize
fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(mask.numpy(), cmap='YlOrRd', vmin=0, vmax=1, aspect='equal')

# Draw quadrant boundaries
ax.axhline(N_VISUAL_TOKENS - 0.5, color='black', linewidth=2)
ax.axvline(N_VISUAL_TOKENS - 0.5, color='black', linewidth=2)

# Label quadrants
ax.text(N_VISUAL_TOKENS * 0.5, N_VISUAL_TOKENS * 0.5, 'V -> V\n(bidirectional)',
        fontsize=14, ha='center', va='center', color='white', fontweight='bold',
        bbox=dict(boxstyle='round', fc=ACCENT, alpha=0.7))
ax.text(N_VISUAL_TOKENS + N_TEXT * 0.5, N_VISUAL_TOKENS * 0.5, 'V -> T\n(blocked)',
        fontsize=14, ha='center', va='center', color='black', fontweight='bold',
        bbox=dict(boxstyle='round', fc='white', alpha=0.7))
ax.text(N_VISUAL_TOKENS * 0.5, N_VISUAL_TOKENS + N_TEXT * 0.5, 'T -> V\n(full access)',
        fontsize=14, ha='center', va='center', color='white', fontweight='bold',
        bbox=dict(boxstyle='round', fc=TEAL, alpha=0.7))
ax.text(N_VISUAL_TOKENS + N_TEXT * 0.5, N_VISUAL_TOKENS + N_TEXT * 0.5, 'T -> T\n(causal)',
        fontsize=14, ha='center', va='center', color='white', fontweight='bold',
        bbox=dict(boxstyle='round', fc=BLUE, alpha=0.7))

# Axis labels
ax.set_xlabel('Attends TO (Key)', fontsize=12)
ax.set_ylabel('Attends FROM (Query)', fontsize=12)

# Tick labels for key positions
tick_pos = [0, 97, 195, 196, 197, 198, 199, 200, 201, 202]
tick_labels = ['img_0', 'img_97', 'img_195', 'pick', 'up', 'the', 'red', 'cup', 'from', 'table']
ax.set_xticks(tick_pos)
ax.set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=8)
ax.set_yticks(tick_pos)
ax.set_yticklabels(tick_labels, fontsize=8)

ax.set_title(f'VLM Attention Mask ({N_TOTAL}x{N_TOTAL})',
             fontsize=14, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"VLM Attention Mask ({N_TOTAL}x{N_TOTAL}):")
print(f"  V->V (top-left):     Bidirectional — all patches see all patches")
print(f"  V->T (top-right):    Blocked — image doesn't peek at text (prefix LM)")
print(f"  T->V (bottom-left):  Full access — every text token sees ALL patches")
print(f"  T->T (bottom-right): Causal — autoregressive text generation")
print(f"\nThe T->V quadrant is where cross-modal grounding happens!")
print(f"'red' can attend to every image patch to find the red regions.")

---
## Step 5: Building a Mini-VLM from Scratch

Now we build a complete, working mini-VLM. It combines:
1. **PatchEmbedding** — image → patch tokens
2. **Text embedding** — words → token vectors
3. **Linear projection** — align ViT dim → shared dim
4. **Transformer blocks** — cross-modal self-attention
5. **Output head** — predict next text token

This is structurally identical to PaliGemma, just ~1200x smaller.

In [ ]:
# Core building blocks (compact versions from Parts 3 & 4)

class PatchEmbedding(nn.Module):
    """Image → patch tokens via Conv2d."""
    def __init__(self, img_size=224, patch_size=16, in_channels=3, d_model=256):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, d_model,
                              kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)       # [B, d_model, H/P, W/P]
        x = x.flatten(2)       # [B, d_model, n_patches]
        x = x.transpose(1, 2)  # [B, n_patches, d_model]
        return x


class MultiHeadAttention(nn.Module):
    """Multi-head self-attention with optional mask."""
    def __init__(self, d_model=256, n_heads=4):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, N, D = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.n_heads, self.d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # [3, B, heads, N, d_head]
        Q, K, V = qkv[0], qkv[1], qkv[2]

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_head)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        weights = F.softmax(scores, dim=-1)
        out = (weights @ V).transpose(1, 2).reshape(B, N, D)
        return self.proj(out), weights


class TransformerBlock(nn.Module):
    """Pre-LN Transformer block with attention + FFN."""
    def __init__(self, d_model=256, n_heads=4, d_ff=1024):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        attn_out, weights = self.attn(self.norm1(x), mask)
        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x, weights


print("Building blocks defined:")
print("  PatchEmbedding — Conv2d trick for image patching")
print("  MultiHeadAttention — fused QKV with optional mask")
print("  TransformerBlock — Pre-LN with residual connections")

In [ ]:
class MiniVLM(nn.Module):
    """A minimal Vision-Language Model.

    Architecture (mirrors PaliGemma):
      Image → PatchEmbedding (d_vit) → Linear Projection (d_model) → [visual tokens]
      Text  → Embedding (d_model) → [text tokens]
      [visual | text] → Transformer Blocks → Output Head → next-token logits
    """
    def __init__(self, img_size=224, patch_size=16, in_channels=3,
                 d_vit=128, d_model=256, n_heads=4, n_layers=4, d_ff=1024,
                 vocab_size=50, max_text_len=20):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2  # 196

        # Vision encoder
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, d_vit)
        self.vis_pos_embed = nn.Parameter(torch.randn(1, self.n_patches, d_vit) * 0.02)

        # Projection bridge: ViT dim → shared dim
        self.projection = nn.Linear(d_vit, d_model)

        # Text encoder
        self.text_embed = nn.Embedding(vocab_size, d_model)
        self.text_pos_embed = nn.Parameter(torch.randn(1, max_text_len, d_model) * 0.02)

        # Shared Transformer
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

        # Output head (predict next text token)
        self.head = nn.Linear(d_model, vocab_size)

    def build_vlm_mask(self, n_vis, n_txt, device):
        """Build the VLM attention mask (PaliGemma-style prefix LM)."""
        N = n_vis + n_txt
        mask = torch.zeros(N, N, device=device)
        # Visual: bidirectional
        mask[:n_vis, :n_vis] = 1.0
        # Text→Visual: full access
        mask[n_vis:, :n_vis] = 1.0
        # Text→Text: causal
        for i in range(n_txt):
            for j in range(i + 1):
                mask[n_vis + i, n_vis + j] = 1.0
        return mask.unsqueeze(0).unsqueeze(0)  # [1, 1, N, N] for broadcasting

    def forward(self, image, text_ids):
        B = image.shape[0]
        n_txt = text_ids.shape[1]

        # Vision path: image → patches → projection
        vis_tokens = self.patch_embed(image)           # [B, 196, d_vit]
        vis_tokens = vis_tokens + self.vis_pos_embed   # add position
        vis_tokens = self.projection(vis_tokens)       # [B, 196, d_model]

        # Text path: token IDs → embeddings
        txt_tokens = self.text_embed(text_ids)         # [B, n_txt, d_model]
        txt_tokens = txt_tokens + self.text_pos_embed[:, :n_txt]  # add position

        # Concatenate: [visual | text]
        x = torch.cat([vis_tokens, txt_tokens], dim=1)  # [B, 196+n_txt, d_model]

        # Build attention mask
        mask = self.build_vlm_mask(self.n_patches, n_txt, image.device)

        # Transformer blocks
        all_weights = []
        for block in self.blocks:
            x, weights = block(x, mask)
            all_weights.append(weights)

        x = self.norm(x)

        # Output logits (only for text positions)
        text_output = x[:, self.n_patches:]  # [B, n_txt, d_model]
        logits = self.head(text_output)       # [B, n_txt, vocab_size]

        return logits, all_weights, x


# Build the mini-VLM
VOCAB_SIZE = 50
vlm = MiniVLM(
    img_size=224, patch_size=16, in_channels=3,
    d_vit=128, d_model=256, n_heads=4, n_layers=4, d_ff=1024,
    vocab_size=VOCAB_SIZE, max_text_len=20,
)

total_params = sum(p.numel() for p in vlm.parameters())
print(f"Mini-VLM Architecture:")
print(f"  d_vit: 128, d_model: 256, heads: 4, layers: 4, d_ff: 1024")
print(f"  Vocab: {VOCAB_SIZE}, Patches: {vlm.n_patches}")
print(f"  Total parameters: {total_params:,} ({total_params/1e6:.2f}M)")
print(f"")
print(f"Component breakdown:")
for name, module in [('Patch Embedding', vlm.patch_embed),
                     ('Projection', vlm.projection),
                     ('Text Embedding', vlm.text_embed),
                     ('Transformer (4 blocks)', vlm.blocks),
                     ('Output Head', vlm.head)]:
    p = sum(p.numel() for p in module.parameters())
    print(f"  {name:>25s}: {p:>10,} params")

In [ ]:
# Forward pass with our robot scene + instruction
img_tensor = torch.tensor(scene).permute(2, 0, 1).unsqueeze(0)  # [1, 3, 224, 224]

# Simple vocabulary for the demo
instruction = "pick up the red cup from table"
words = instruction.split()
vocab_words = sorted(set(words))
word2id = {w: i for i, w in enumerate(vocab_words)}
text_ids = torch.tensor([[word2id[w] for w in words]])  # [1, 7]

print(f"Input:")
print(f"  Image: {list(img_tensor.shape)}")
print(f"  Text:  '{instruction}' → IDs: {text_ids[0].tolist()}")
print(f"  Vocab: {word2id}")
print()

# Run forward pass
with torch.no_grad():
    logits, all_weights, full_output = vlm(img_tensor, text_ids)

print(f"Forward pass shapes:")
print(f"  Patches:       [1, 196, 128]  → image split into 14x14 patches")
print(f"  After proj:    [1, 196, 256]  → projected to shared d_model")
print(f"  Text embeds:   [1, 7, 256]    → 7 word embeddings")
print(f"  Concatenated:  [1, 203, 256]  → unified sequence")
print(f"  After transformer: {list(full_output.shape)}")
print(f"  Text logits:   {list(logits.shape)}  → next-token predictions")
print(f"")
print(f"The model processes vision and language in ONE transformer.")
print(f"Every text token can attend to every image patch via cross-modal attention.")

In [ ]:
# Cross-modal attention visualization (the money plot)
# Extract attention from last layer, average across heads
last_layer_attn = all_weights[-1]  # [1, n_heads, 203, 203]
avg_attn = last_layer_attn[0].mean(dim=0)  # [203, 203] — average over heads

# For each text word, show attention over the 14x14 image grid
PATCH_SIZE = 16
n_vis = 196
focus_words = [("pick", 0), ("red", 3), ("cup", 4)]
focus_colors = [ACCENT, BLUE, TEAL]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (word, word_idx), c in zip(axes, focus_words, focus_colors):
    # Attention from text token to image patches
    text_pos = n_vis + word_idx  # position in the 203-token sequence
    text_to_image = avg_attn[text_pos, :n_vis].detach().numpy()  # [196]
    attn_map = text_to_image.reshape(14, 14)

    ax.imshow(scene)
    ax.imshow(np.kron(attn_map, np.ones((PATCH_SIZE, PATCH_SIZE))),
              alpha=0.65, cmap='hot', vmin=0)
    ax.set_title(f'"{word}" → image patches', fontsize=14, color=c, fontweight='bold')
    ax.axis('off')

plt.suptitle('Cross-Modal Attention: Text Words Attend to Image Regions',
             fontsize=15, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

print("Note: With random weights, these patterns are meaningless.")
print("But the STRUCTURE is what matters — the wiring is correct:")
print("  - 'pick' CAN attend to gripper patches")
print("  - 'red' CAN attend to red-colored patches")
print("  - 'cup' CAN attend to cup-shaped patches")
print("\nTraining on millions of image-text pairs fills these connections with meaning.")
print("The architecture enables grounding; the data teaches it.")

In [ ]:
# Full 203x203 attention matrix visualization
fig, ax = plt.subplots(figsize=(10, 9))

attn_full = avg_attn.detach().numpy()  # [203, 203]
im = ax.imshow(attn_full, cmap='YlOrRd', vmin=0, aspect='equal')

# Draw quadrant boundaries
ax.axhline(n_vis - 0.5, color='black', linewidth=2)
ax.axvline(n_vis - 0.5, color='black', linewidth=2)

# Label quadrants
ax.text(n_vis * 0.5, n_vis * 0.5, 'V -> V\n(bidirectional)',
        fontsize=14, ha='center', va='center', color='white', fontweight='bold',
        bbox=dict(boxstyle='round', fc=ACCENT, alpha=0.7))
ax.text(n_vis + N_TEXT * 0.5, n_vis * 0.5, 'V -> T\n(masked)',
        fontsize=14, ha='center', va='center', color='black', fontweight='bold',
        bbox=dict(boxstyle='round', fc='white', alpha=0.7))
ax.text(n_vis * 0.5, n_vis + N_TEXT * 0.5, 'T -> V\n(cross-modal)',
        fontsize=14, ha='center', va='center', color='white', fontweight='bold',
        bbox=dict(boxstyle='round', fc=TEAL, alpha=0.7))
ax.text(n_vis + N_TEXT * 0.5, n_vis + N_TEXT * 0.5, 'T -> T\n(causal)',
        fontsize=14, ha='center', va='center', color='white', fontweight='bold',
        bbox=dict(boxstyle='round', fc=BLUE, alpha=0.7))

# Axis labels
tick_pos = [0, 97, 195, 196, 197, 198, 199, 200, 201, 202]
tick_labels_x = ['img_0', 'img_97', 'img_195'] + words
ax.set_xticks(tick_pos)
ax.set_xticklabels(tick_labels_x, rotation=45, ha='right', fontsize=8)
ax.set_yticks(tick_pos)
ax.set_yticklabels(tick_labels_x, fontsize=8)
ax.set_xlabel('Attends TO (Key)', fontsize=12)
ax.set_ylabel('Attends FROM (Query)', fontsize=12)

plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title(f'Full VLM Attention ({N_TOTAL}x{N_TOTAL}) — Last Layer, Head Average',
             fontsize=14, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"The 4 quadrants of VLM attention:")
print(f"  V->V (top-left):     Image patches attend to each other")
print(f"  V->T (top-right):    Masked — image can't peek at text (prefix LM)")
print(f"  T->V (bottom-left):  Cross-modal grounding — THIS is the magic")
print(f"  T->T (bottom-right): Causal text generation")
print(f"\nThe T->V quadrant is where 'red' learns to attend to red pixels,")
print(f"'cup' learns to attend to cup patches, and 'pick' finds the gripper.")

---
## Step 6: PaliGemma — The Real VLM Behind pi0

Our mini-VLM is structurally identical to **PaliGemma**, the VLM used inside pi0. The difference? Scale.

PaliGemma combines:
- **SigLIP-So400M** — a 400M-parameter vision encoder trained on 4B image-text pairs
- **Linear projection** — 2.4M parameters to bridge dimensions
- **Gemma 2B** — a 2-billion parameter language model trained on trillions of tokens

In [ ]:
# PaliGemma architecture breakdown
print("="*65)
print("PaliGemma Architecture (the VLM inside pi0)")
print("="*65)
print()

components = [
    ("SigLIP-So400M (vision)", 400_000_000, "Frozen", "4B image-text pairs"),
    ("Linear Projection", 2_400_000, "Trained", "—"),
    ("Gemma 2B (language)", 2_000_000_000, "Frozen", "Trillions of tokens"),
]

print(f"{'Component':>30s} | {'Params':>12s} | {'Status':>8s} | {'Training Data':>25s}")
print("-" * 85)
total = 0
for name, params, status, data in components:
    total += params
    print(f"{name:>30s} | {params/1e6:>10.1f}M | {status:>8s} | {data:>25s}")
print("-" * 85)
print(f"{'TOTAL':>30s} | {total/1e6:>10.1f}M |")

print(f"\n\nOur mini-VLM vs PaliGemma:")
print(f"  Mini-VLM:   {total_params:>12,} params ({total_params/1e6:.2f}M)")
print(f"  PaliGemma:  {total:>12,} params ({total/1e6:.0f}M)")
print(f"  Scale:      {total / total_params:.0f}x larger")
print(f"\nSame architecture, same attention mask, same prefix-LM pattern.")
print(f"The difference is {total / total_params:.0f}x more parameters + massive pre-training data.")

In [ ]:
# Log-scale parameter count comparison: from TinyCNN to PaliGemma
models = [
    ('TinyCNN\n(Part 2)', 50_000, WARM),
    ('TextGRU\n(Part 1)', 130_000, WARM),
    ('Mini-VLA\n(Part 2)', 500_000, WARM),
    ('Our Mini-VLM\n(this notebook)', total_params, TEAL),
    ('ViT-Base', 86_000_000, BLUE),
    ('SigLIP-So400M', 400_000_000, BLUE),
    ('Gemma 2B', 2_000_000_000, PURPLE),
    ('PaliGemma\n(full VLM)', 2_400_000_000, ACCENT),
]

fig, ax = plt.subplots(figsize=(14, 6))

names = [m[0] for m in models]
params = [m[1] for m in models]
colors = [m[2] for m in models]

bars = ax.bar(range(len(models)), params, color=colors, edgecolor='white', linewidth=1.5)
ax.set_yscale('log')
ax.set_xticks(range(len(models)))
ax.set_xticklabels(names, fontsize=9, ha='center')
ax.set_ylabel('Parameters (log scale)', fontsize=12)
ax.set_title('Parameter Counts: From Toy Models to Production VLMs',
             fontsize=14, color=ACCENT, fontweight='bold')

# Label each bar
for i, (bar, p) in enumerate(zip(bars, params)):
    if p >= 1e9:
        label = f'{p/1e9:.1f}B'
    elif p >= 1e6:
        label = f'{p/1e6:.0f}M'
    else:
        label = f'{p/1e3:.0f}K'
    ax.text(i, p * 1.5, label, ha='center', fontsize=10, fontweight='bold', color=colors[i])

ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(1e4, 1e11)

# Bracket for "bootcamp models" vs "production models"
ax.axvline(3.5, color='gray', linestyle='--', alpha=0.5)
ax.text(1.5, 5e9, 'Bootcamp models', fontsize=11, ha='center', color='gray', style='italic')
ax.text(5.5, 5e9, 'Production models', fontsize=11, ha='center', color='gray', style='italic')

plt.tight_layout()
plt.show()

print(f"Our mini-VLM: {total_params/1e6:.1f}M params — enough to learn the STRUCTURE.")
print(f"PaliGemma: 2,400M params — enough to learn the WORLD.")
print(f"\nPre-training gives PaliGemma:")
print(f"  - Object recognition (knows what a cup looks like)")
print(f"  - Language understanding (knows 'pick up' means grasp + lift)")
print(f"  - Cross-modal grounding ('red cup' → red cup pixels)")
print(f"  - Spatial reasoning ('on the table' → bottom of image)")
print(f"All for FREE — no robot data needed.")

---
## Step 7: From VLM to VLA — The Last Mile

A VLM understands images and text. A **VLA** (Vision-Language-Action model) additionally outputs robot actions. How do we get from VLM to VLA?

Two dominant approaches have emerged:

In [ ]:
# Two approaches: tokenization vs flow matching
print("="*65)
print("From VLM to VLA: Two Approaches")
print("="*65)
print()

print("Approach 1: RT-2 / OpenVLA — Tokenize Actions")
print("-" * 50)
print("  VLM output: 'pick up the red cup'")
print("  Action output: '128 64 200 55 90 12 1'  (discretized motor commands)")
print("  Method: Treat actions as text tokens → next-token prediction")
print("  Pro: Simple — just extend the vocabulary")
print("  Con: Discretization loses precision, bad for fine manipulation")
print()

print("Approach 2: pi0 — Flow Matching Action Expert")
print("-" * 50)
print("  VLM backbone: PaliGemma (FROZEN) — provides understanding")
print("  Action expert: Separate model (TRAINED) — generates actions")
print("  Method: Flow matching (continuous) instead of tokenization (discrete)")
print("  Pro: Continuous actions → smooth, precise robot motion")
print("  Con: More complex architecture (two models)")
print()

print("="*65)
print("The pi0 Recipe:")
print("="*65)
print()
print("  1. PaliGemma VLM (frozen)")
print("     - SigLIP encodes the camera image → 196 visual tokens")
print("     - Gemma processes [visual | text] with cross-modal attention")
print("     - Outputs: rich, grounded representations")
print()
print("  2. Action Expert (trainable)")
print("     - Takes VLM representations + noisy action guess")
print("     - Flow matching: predicts velocity field to denoise")
print("     - Outputs: continuous action trajectory (joint angles + gripper)")
print()
print("  3. Training")
print("     - Only the action expert trains on robot demonstrations")
print("     - PaliGemma's world knowledge comes for free")
print("     - 10K demonstrations → dexterous manipulation")
print()
print("This is what we build in the next lecture. Everything you learned here")
print("— SigLIP, projection, VLM assembly, cross-modal attention —")
print("is the foundation that pi0 builds on.")

---
## Summary

| Component | What it does | Key insight |
|-----------|-------------|-------------|
| **SigLIP** | Aligns image and text embeddings | Sigmoid per-pair loss works with small batches |
| **Projection Bridge** | Maps ViT dim → LLM dim | Just a linear layer — ViT features are already rich |
| **VLM Assembly** | [visual \| text] → one sequence | Cross-modal attention emerges naturally |
| **Cross-Modal Attention** | Text words attend to image patches | "red" → red pixels, "cup" → cup shape |
| **PaliGemma** | SigLIP + Gemma 2B = production VLM | 2.4B params of pre-trained world knowledge |

### The Key Insight

Pre-trained VLMs bring **world knowledge for free**. They already know what a cup looks like, what "pick up" means, and where objects sit on tables. Robots only need to learn the **last mile** — turning that understanding into motor commands.

That's exactly what pi0 does: freeze PaliGemma, train only the action expert.

---

## Exercises

1. **Temperature ablation:** Modify the SigLIP training loop to try temperatures of 0.1, 1.0, and 10.0. How does temperature affect the final similarity matrix? Plot all three side by side. Why does lower temperature produce sharper diagonals?

2. **Larger contrastive batch:** Increase the SigLIP training from 8 to 32 pairs. Does alignment improve? How does training speed change? Compare the final loss values.

3. **Pre-trained ViT features:** Replace our PatchEmbedding with a real pre-trained ViT from torchvision (`torchvision.models.vit_b_16(weights='DEFAULT')`). Extract patch features and feed them into the mini-VLM. How do the cross-modal attention maps change compared to random features?

4. **Challenge — Add an action head:** Extend MiniVLM to output continuous robot actions (6-DoF: x, y, z, roll, pitch, yaw). Add an `action_head` MLP that takes the final text token's representation and outputs 6 continuous values. This mirrors the VLM→VLA transition that pi0 makes.